# IndicBERT v2 — Multilingual Intent Classification (English / Hinglish / Hindi)
Data Sprint 2.0: 56-class intent classification. Fine-tunes `ai4bharat/IndicBERTv2-MLM-only` on 2x T4 (Trainer auto-uses `DataParallel` across both GPUs, fp16).

In [7]:
!pip install -q -U transformers accelerate scikit-learn

In [8]:
import os, re, glob, unicodedata, random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback, set_seed)

SEED = 42
MAX_LEN = 64
EPOCHS = 8
LR = 3e-5
PER_DEVICE_BS = 32
NUM_LABELS = 56
OUT_DIR = "/kaggle/working/indicbert_v2_out" if os.path.exists("/kaggle") else "./indicbert_v2_out"

set_seed(SEED)
n_gpu = torch.cuda.device_count()
print("GPUs visible:", n_gpu)
for i in range(n_gpu):
    print(i, torch.cuda.get_device_name(i))
assert n_gpu >= 2, "Expected 2 GPUs (enable 'GPU T4 x2' in the accelerator settings)"
print("Effective train batch size:", PER_DEVICE_BS * n_gpu)

GPUs visible: 2
0 Tesla T4
1 Tesla T4
Effective train batch size: 64


In [9]:
def find_data_dir():
    for root in ["/kaggle/input", ".", "/content", "/mnt/user-data/uploads"]:
        hits = glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError("train.csv not found")

DATA_DIR = find_data_dir()
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_df.head()

,Unnamed: 0,example_id,query_text,target,script,transliterated_text
0,0,100001,मैं चाहता हूं कि इस घटना को मेरे कैलेंडर से हट...,29,devanagari,main chaahata huun kii iss ghatana coo merey c...
1,1,100002,आधा करने के लिए कम मात्रा,45,devanagari,aadha karane key liye cum matra
2,2,100003,kripaya rasoeeghar ki battiyaan band kar den,39,latin,kripaya rasoeeghar ki battiyaan band kar den
3,3,100004,vaikyum on karo,33,latin,vaikyum on karo
4,4,100005,ढाई घंटे के बाद घर को ऑटो लॉक करें,12,devanagari,dhaaee ghante key baad ghar coo auto lock karein


## Quick EDA

In [10]:
def script_of(t):
    t = str(t)
    dev = sum('\u0900' <= c <= '\u097F' for c in t)
    lat = sum(c.isascii() and c.isalpha() for c in t)
    if dev and lat: return "mixed"
    if dev: return "devanagari"
    return "latin"

    df["script"] = df["query_text"].map(script_of)
    print(name, df["script"].value_counts(normalize=True).round(3).to_dict())

print("Train length (chars):", train_df["query_text"].str.len().describe().round(1).to_dict())
print("Class counts: min", train_df["target"].value_counts().min(), "max", train_df["target"].value_counts().max())
print("Out-of-scope (55) share:", (train_df["target"] == 55).mean().round(4))

## Preprocessing, split, tokenization
Light normalization only (NFKC, whitespace) — no transliteration, so the model sees Hinglish directly in Hinglish.

## Model and training
Single process, two T4s via `DataParallel`; fp16 for speed.

In [ ]:
import random
import torch.nn.functional as F
from torch.utils.data import WeightedRandomSampler

MODEL_NAME = "l3cube-pune/hing-roberta-mixed"
POWER = 0.5
LABEL_SMOOTHING = 0.05
HEAD_LR = 1e-3
N_NOISY = 1
NOISE_P = 0.2
SHARE_ORIG_LATIN = 0.32
SHARE_AUG_LATIN = 0.33
SHARE_DEV = 0.30
SHARE_MIXED = 0.05

random.seed(SEED)

def find_data_dir():
    for root in ["/kaggle/input", ".", "/content", "/mnt/user-data/uploads"]:
        hits = glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError("train.csv not found")

DATA_DIR = find_data_dir()
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

def clean(t):
    t = unicodedata.normalize("NFKC", str(t))
    t = re.sub(r"\s+", " ", t).strip()
    return t

    df["text"] = df["query_text"].map(clean)

tr_df = train_df.reset_index(drop=True).copy()
tr_df["src"] = "orig"

RULES = [
    ("aa", "a"), ("a", "aa"), ("ee", "i"), ("i", "ee"), ("oo", "u"), ("u", "oo"),
    ("ii", "i"), ("uu", "u"), ("kh", "k"), ("sh", "s"), ("s", "sh"), ("w", "v"),
    ("v", "w"), ("ch", "c"), ("ph", "f"), ("f", "ph"), ("th", "t"), ("dh", "d"),
    ("ai", "ae"), ("ey", "e"), ("ye", "e"), ("hh", "h"), ("n", "m"),
]

def noisy_word(w):
    if len(w) < 3 or random.random() > NOISE_P:
        return w
    options = [(a, b) for a, b in RULES if a in w]
    if not options:
        return w
    a, b = random.choice(options)
    return w.replace(a, b, 1)

def noisy(text):
    return " ".join(noisy_word(w) for w in text.split())

dev = tr_df[(tr_df["script"] == "devanagari") & tr_df["transliterated_text"].notna()]
aug_rows = []
for _, r in dev.iterrows():
    base = clean(r["transliterated_text"]).lower()
    if not base:
        continue
    variants = {base}
    for _ in range(N_NOISY):
        variants.add(noisy(base))
    for v in variants:
        aug_rows.append({"text": v, "target": r["target"], "script": "latin", "src": "aug"})

aug_df = pd.DataFrame(aug_rows)
tr_df = pd.concat([tr_df[["text", "target", "script", "src"]], aug_df], ignore_index=True)
tr_df = tr_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print("train:", len(tr_df), "augmented added:", len(aug_df))
print(tr_df.groupby(["script", "src"]).size())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

def to_ds(df, labeled=True):
    cols = {"text": df["text"].tolist()}
    if labeled:
        cols["labels"] = df["target"].astype(int).tolist()
    return Dataset.from_dict(cols).map(tok, batched=True, remove_columns=["text"])

train_ds = to_ds(tr_df)

lens = [len(x) for x in train_ds["input_ids"]]
print("Token len p95/p99/max:", np.percentile(lens, 95), np.percentile(lens, 99), max(lens))

g_orig_lat = ((tr_df["script"] == "latin") & (tr_df["src"] == "orig")).values
g_aug_lat = (tr_df["src"] == "aug").values
g_dev = ((tr_df["script"] == "devanagari") & (tr_df["src"] == "orig")).values
g_mixed = ((tr_df["script"] == "mixed") & (tr_df["src"] == "orig")).values

sample_w = np.zeros(len(tr_df))
for mask, share in [(g_orig_lat, SHARE_ORIG_LATIN), (g_aug_lat, SHARE_AUG_LATIN), (g_dev, SHARE_DEV), (g_mixed, SHARE_MIXED)]:
    if mask.sum() > 0:
        sample_w[mask] = share / mask.sum()
sampler_weights = torch.tensor(sample_w, dtype=torch.double)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
print("params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

train_labels = np.array(train_ds["labels"])
counts = np.maximum(np.bincount(train_labels, minlength=NUM_LABELS), 1).astype(float)
print("max/min count ratio:", counts.max() / counts.min())

w = (counts.max() / counts) ** POWER
w[55] = 1.0
w = w * counts.sum() / (w * counts).sum()
class_weights = torch.tensor(w, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = F.cross_entropy(
            logits.float(),
            labels,
            weight=class_weights.to(logits.device),
            label_smoothing=LABEL_SMOOTHING,
        )
        return (loss, outputs) if return_outputs else loss

    def _get_train_sampler(self, *args, **kwargs):
        return WeightedRandomSampler(sampler_weights, num_samples=len(sampler_weights), replacement=True)

    def create_optimizer(self):
        if self.optimizer is None:
            head = [p for n, p in self.model.named_parameters() if "classifier" in n]
            base = [p for n, p in self.model.named_parameters() if "classifier" not in n]
            self.optimizer = torch.optim.AdamW(
                [
                    {"params": base, "lr": self.args.learning_rate},
                    {"params": head, "lr": HEAD_LR},
                ],
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer

In [ ]:
args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BS,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=True,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=50,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
)

print("Trainer n_gpu:", trainer.args.n_gpu)
trainer.train()

trainer.save_model(f"{OUT_DIR}/best")
tokenizer.save_pretrained(f"{OUT_DIR}/best")


In [15]:
import os
import shutil
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

best_dir = "indicbert_v2_out/best"

if os.path.isdir(best_dir):
    zip_path = shutil.make_archive(
        "indicbert_best",
        "zip",
        root_dir="indicbert_v2_out",
        base_dir="best"
    )

    print("ZIP created:", zip_path)
    print("Size:", round(os.path.getsize(zip_path) / (1024**2), 2), "MB")

    display(FileLink("indicbert_best.zip"))
else:
    print("Error: best directory not found!")

ZIP created: /kaggle/working/indicbert_best.zip
Size: 986.37 MB


/kaggle/working/indicbert_best.zip

## Next steps to improve
- Ensemble with `ai4bharat/IndicBERTv2-SS` or `Calibrated LinearSVC` via log-space geometric gating.
- Class 55 (Out-of-Scope): tune a max-softmax confidence threshold (`OOS_CONF`) on validation to handle low-confidence predictions.